## Importing Libraries

In [ ]:
from ollama import chat
import glob
from tqdm import tqdm
import os

## Setting up files

In [21]:
GENERATION_MODEL = "qwen3:1.7b"
FILES_EXTR = glob.glob("../Test_Files/clinical-trial_*.txt")
FILES_CONV = glob.glob("../Test_Files/clinical-trial-extracted*.txt")

PROMPT_EXTR_FILE = "./prompts/criteria-extraction_prompt.txt"
OUTPUT_EXTR_DIR = "./llm-outputs/criteria-extraction/"
OUTPUT_EXTR_FILE = "experiment"

PROMPT_CONV_FILE = "./prompts/criteria-conversion_prompt.txt"
OUTPUT_CONV_DIR = "./llm-outputs/criteria-conversion/"
OUTPUT_CONV_FILE = "experiment"

print(f"Found the following files for extraction - {FILES_EXTR}")
print(f"Found the following files for conversion - {FILES_CONV}")

Found the following files for extraction - ['../Test_Files\\clinical-trial_e1.txt']
Found the following files for conversion - ['../Test_Files\\clinical-trial-extracted_e1.txt']


## Pre-Processing

In [22]:
pass

## Setting up environment

In [23]:
## Setting evironment
def set_env(prompt_file,output_dir):
    with open(prompt_file,"r", encoding="utf-8") as p:
        prompt_arr = [t.strip() for t in p.readlines() if t.strip()]
        base_prompt = " ".join(prompt_arr)

    os.makedirs(output_dir,exist_ok=True)

    count = 0

    for path in os.listdir(output_dir):
        if os.path.isfile(os.path.join(output_dir, path)):
            count += 1
    
    return base_prompt, count

base_prompt_extr, count_extr_exp = set_env(PROMPT_EXTR_FILE, OUTPUT_EXTR_DIR)

base_prompt_conv, count_conv_exp = set_env(PROMPT_CONV_FILE, OUTPUT_CONV_DIR)
        


## Criteria Extraction
In this first phase the criteria of a given clinical trial are extracted still in natural language to make the conversion easier

In [24]:
pbar = tqdm(total=len(FILES_EXTR), desc="Processing trials for criteria extraction")

for file in FILES_EXTR:
    with open(file,"r", encoding="utf-8") as f:
        text_arr = [t.strip() for t in f.readlines() if t.strip()]
        text = " ".join(text_arr)
        
        print(f"processing file: {file}")
        
        prompt = base_prompt_extr.replace("{{TRIAL_TEXT}}",text)
        
        stream = chat(
            model=GENERATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            stream=True,
            )
        
        llm_output = ""
        for chunk in stream:
            llm_output += chunk["message"]["content"]

        with open(f"{OUTPUT_EXTR_DIR}{OUTPUT_EXTR_FILE}-{count_extr_exp}.txt","w",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_EXTR_FILE}-{count_extr_exp}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Processing trials for criteria extraction:   0%|          | 0/2 [00:33<?, ?it/s]

processing file: ../Test_Files\clinical-trial_e1.txt



Processing trials for criteria extraction: 100%|██████████| 1/1 [01:09<00:00, 69.76s/it]

Saved LLM output on experiment-1




## Criteria Conversion
In this second phase the already extracted criteria in natural language of a given clinical trial will be converted into logical rules that way allowing the deterministic matching of patients with the clinical trial

In [25]:
pbar = tqdm(total=len(FILES_CONV), desc="Processing trials for criteria conversion")

for file in FILES_CONV:
    with open(file,"r", encoding="utf-8") as f:
        text_arr = [t.strip() for t in f.readlines() if t.strip()]
        text = " ".join(text_arr)
        
        print(f"processing file: {file}")
        
        prompt = base_prompt_conv.replace("{{TRIAL_TEXT}}",text)
        
        stream = chat(
            model=GENERATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            stream=True,
            )
        
        llm_output = ""
        for chunk in stream:
            llm_output += chunk["message"]["content"]

        with open(f"{OUTPUT_CONV_DIR}{OUTPUT_CONV_FILE}-{count_conv_exp}.txt","w",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_CONV_FILE}-{count_conv_exp}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Processing trials for criteria conversion:   0%|          | 0/1 [00:00<?, ?it/s]

processing file: ../Test_Files\clinical-trial-extracted_e1.txt


Processing trials for criteria conversion: 100%|██████████| 1/1 [05:41<00:00, 341.87s/it]

Saved LLM output on experiment-1


